# Registro de experimentos

Llevar la cuenta de los experimentos realizados y los resultados obtenidos puede ser una tarea tediosa. Para aliviar esta carga y garantizar un buen control de los recursos empleados, existen plataformas destinadas a ello.

Estas plataformas llevan a cabo 3 aspectos clave:

* **Experiment tracking**: Seguimiento de experimentos y registro de métricas
* **Model Registry**: Registro de modelos y sus distintas versiones de cara a su posterior productivización
* **Experiment analysis**: Hacen disponibles paneles de control para analizar de forma sencilla el rendimiento de los modelos así como compartir con el resto del equipo los recursos.

Entre las plataformas más comunes encontramos:

* MLFlow (https://mlflow.org/) posiblemente de las primera plataformas en este ámbito
* CometML (https://www.comet.com/) fácil de usar y con una curva de aprendizaje mínima
* Weights & Biases (https://wandb.ai/site)

Donde también destacan las soluciones propietarias de las plataformas nube más conocidas:

* Azure Machine Learning
* Amazon Sagemaker
* Vertex AI (Google)

## Comet ML

A continuación veremos un ejemplo sencillo empleando la API KEY de Comet ML (https://www.comet.com/). Deberéis registraros (podéis usar vuestro usuario de Github) y obtener la API en su página principal una vez dentro.

Tendréis que instalar también el SDK que nos permite realizar la comunicación entre ambos:
```
pip install comet-ml
```
O
```
uv add comet-ml
```

In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [3]:
import os
from comet_ml import Experiment
          
experiment = Experiment(
  api_key=os.environ.get("COMET_APIKEY"),
  project_name="prueba",
  workspace="tb-iraitz"
)

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/tb-iraitz/prueba/e4478d3bdfd94ccc977d3acdaa92c811



Si os aparece que el workspace no existe, deberéis crear uno mediante la web de comet.

![workspace](img/workspace.png)

`experiment` es donde podemos informar de los pasos que vamos dando en nuestros experimentos.

In [4]:
# Report multiple hyperparameters using a dictionary:
hyper_params = {
    "learning_rate": 0.5,
    "steps": 100000,
    "batch_size": 50,
}
experiment.log_parameters(hyper_params)

# Or report single hyperparameters:
hidden_layer_size = 50
experiment.log_parameter("hidden_layer_size", hidden_layer_size)

# Long any time-series metrics:
train_accuracy = 3.14
experiment.log_metric("accuracy", train_accuracy, step=0)

In [5]:
experiment.log_metric("accuracy", 3.12, step=1)

Y una vez hallamos acabado, cerrar el experimento.

In [6]:
experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : head_hamster_2217
COMET INFO:     url                   : https://www.comet.com/tb-iraitz/prueba/e4478d3bdfd94ccc977d3acdaa92c811
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     accuracy [2] : (3.12, 3.14)
COMET INFO:   Parameters:
COMET INFO:     batch_size        : 50
COMET INFO:     hidden_layer_size : 50
COMET INFO:     learning_rate     : 0.5
COMET INFO:     steps             : 100000
COMET INFO:   Uploads:
COMET INFO:     environment details      : 1
COMET INFO:     filename                 : 1
COMET INFO:     git metadata             : 1
COMET INFO:     git-patch (uncompressed) : 1 (288.24 KB)
COMET INFO:     installed packages      

Vemos que esto fijará un proyecto con los datos que hayamos compartido. Si estamos trabajando en equipo, podremos registrar las pruebas tanto nuestras como de nuestros compañeros. 

![exp](img/experiment.png)

Un ejemplo más serio...

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

random_state = 42

experiment = Experiment(
  api_key=os.environ.get("COMET_APIKEY"),
  project_name="prueba-seria",
  workspace="tb-iraitz",
  log_code=True
)

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/tb-iraitz/prueba-seria/468bdf92d7b24e3ca41e0d7f4c56faf7



COMET INFO: Artifact 'tb-iraitz/cancer-data:1.0.0' has been fully uploaded successfully


Podemos seguir con nuestros ejercicios de forma habitual, creando funciones de evaluación con las métricas de interés.

In [8]:
def evaluate(y_test, y_pred):
    return {
        "f1": f1_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
    }

Cargando los datos y separando entre train y test.

In [9]:
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=random_state
)

Instanciando un modelo y entrenándolo.

In [ ]:
clf = RandomForestClassifier()

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


Aunque para registrar nuestro proceso si que deberemos informar de en qué fase se han registrado las métricas de entrenamiento.

In [11]:
with experiment.train():
    # Entrenamos
    clf.fit(X_train, y_train)
    y_train_pred = clf.predict(X_train)
    metrics = evaluate(y_train, y_train_pred)

    # Y registramos el progreso
    experiment.log_metrics(metrics)
    experiment.log_metric("score", clf.score(X_train, y_train))

Y podemos hacer lo mismo con los eventos de validación o test.

In [12]:
with experiment.test():
    # Predecimos
    y_test_pred = clf.predict(X_test)
    metrics = evaluate(y_test, y_test_pred)

    # Registramos el progreso
    experiment.log_metrics(metrics)

Podemos cambiar a otro modelo, un SVC por ejemplo.

In [13]:
from sklearn.svm import SVC

clf = SVC()

Y repetir el proceso.

In [14]:
with experiment.train():
    # Entrenamos
    clf.fit(X_train, y_train)
    y_train_pred = clf.predict(X_train)
    metrics = evaluate(y_train, y_train_pred)

    # Y registramos el progreso
    experiment.log_metrics(metrics)
    experiment.log_metric("score", clf.score(X_train, y_train))

In [15]:
with experiment.test():
    # Predecimos
    y_test_pred = clf.predict(X_test)
    metrics = evaluate(y_test, y_test_pred)

    # Registramos el progreso
    experiment.log_metrics(metrics)

Sin embargo, los datos se guardarán como parte del mismo experimento siendo realizado ya que son parte del **mismo experimento**. Normalmente jugamos con distintas parametrizaciones del modelo, y por eso nos interesa saber qué configuración dió mejor resultado, por ejemplo en un GridSearch.

![experiments](img/experiments.png)

Si nos vamos a uno de esos experimentos, veremos mucha información que está siendo registrada de cara a poder replicar estos experimentos en el futuro.

![instance](img/expinst.png)

Aunque puede que falten cosas si es la reproducibilidad lo que nos preocupa.

## Artefactos

Podemos también registrar artefactos: conjuntos de datos concreto o modelos a almacenar, dentro del mismo contexto del experimento.

In [16]:
cancer = load_breast_cancer(as_frame=True)
cancer.data

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


In [17]:
from comet_ml import Artifact

cancer.data.to_csv("data.csv")

artifact = Artifact(name="cancer-data", artifact_type="dataset")
artifact.add("data.csv")
experiment.log_artifact(artifact)

COMET INFO: Artifact 'cancer-data' version 1.0.0 created
COMET INFO: Scheduling the upload of 1 assets: 1 local assets for a size of 119.54 KB, and 0 remote assets (will be linked, not uploaded). This can take some time.
COMET INFO: Artifact 'tb-iraitz/cancer-data:1.0.0' has started uploading asynchronously


LoggedArtifact(artifact_name='cancer-data', artifact_type='dataset', workspace='tb-iraitz', version=Version('1.0.0'), aliases=frozenset(), artifact_tags=frozenset(), version_tags=frozenset(), size=0, source_experiment_key='468bdf92d7b24e3ca41e0d7f4c56faf7')

También podemos registrar los modelos ya entrenados. Existe una librería muy famosa para guardar modelos de manera eficiente gracias a los amigos de :probabl https://joblib.readthedocs.io/en/stable/

In [18]:
import joblib

# Save the model to local filepath
model_filepath = "svc_classifier.joblib"
joblib.dump(clf, model_filepath)

# Log the model to Comet
experiment.log_model(
    name="svc",
    file_or_folder=model_filepath,
    metadata={"framework": "sklearn"},
)
experiment.register_model("svc")

COMET INFO: Successfully registered 'svc', version None in workspace 'tb-iraitz'


Podemos revisar estos artefactos en el entorno para ver que se han registrado de forma correcta.

![model](img/model.png)

Y una vez hallamos acabado la jornada, cerrar el experimento con la garantía de que la información está guardada y es accesible a todo el equipo.

In [19]:
experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : governing_water_823
COMET INFO:     url                   : https://www.comet.com/tb-iraitz/prueba-seria/468bdf92d7b24e3ca41e0d7f4c56faf7
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     test_f1 [2]         : (0.9411764705882353, 0.967032967032967)
COMET INFO:     test_precision [2]  : (0.9072164948453608, 0.9565217391304348)
COMET INFO:     test_recall         : 0.9777777777777777
COMET INFO:     train_f1 [2]        : (0.9371633752244165, 1.0)
COMET INFO:     train_precision [2] : (0.9, 1.0)
COMET INFO:     train_recall [2]    : (0.9775280898876404, 1.0)
COMET INFO:     train_score [2]     : (0.9178403755868545, 1.0)
COMET INFO:   Parameter

Cuando cerréis el proyecto fijaros que Comet se encarga de guardad el código que hemos empleado en nuestras pruebas. Esto aumenta la reproducibilidad aunque si no andamos con ojo con credenciales y otros aspectos sensibles, podemos exponer información sensible fuera de nuestro entorno personal o incluso equipo de trabajo.

Podemos registrar el modelo para que otros miembros de la organización accedan a él e incluso puedan usarlo.

![registry](img/modelregistry.png)

Esto es lo que otra gente está haciendo con HuggingFace cuando nos permiten descargar sus modelos a voluntad. Comet ha invertido bastante en una solución para gestión de LLMs que también puede sernos de utilidad.

https://www.comet.com/opik/